# Load SN1a Garazi

Voilà le fichier CSV pour cette SN 


    FinkID = 313629129605382159, 
    RubinID = 739161397740437575). 

Dedans tu as :
- flux, flux_err : résultat de ma photométrie d'ouverture en électrons
- psfFlux, psfFluxErr : photométrie PSF de Rubin sur l'image de différence
- scienceFlux, scienceFluxErr : photométrie PSF de Rubin sur l'image de science
- mag, mag_err, flux_calib_njy, flux_calib_err_njy : résultat après avoir appliqué la fonction photoCalib de Rubin pour transformer le flux en électrons en flux en nanoJansky ou en magnitude

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from astropy.time import Time
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB97_01_SNIaGarazi"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
BAND_ORDER = "ugrizy"
BAND_COLOR = {"u": "b", "g": "green", "r": "red", "i": "orange", "z": "grey", "y": "k"}

In [ ]:
!ls data_NB97_01_SNIaGarazi

In [ ]:
filename_rubin = f"{DATA_DIR}/lightcurve_tract_2704_i_739161397740437575.csv"
filename_fink = f"{DATA_DIR}/313629129605382159.csv"

In [ ]:
df_r = pd.read_csv(filename_rubin)
df_f = pd.read_csv(filename_fink)

In [ ]:
index_znota = ~df_f["f:xm_legacydr8_zphot"].isna()
redshift_legdr8 = df_f["f:xm_legacydr8_zphot"][index_znota].values[0]
redshifterr_legdr8 = df_f["f:xm_legacydr8_e_zphot"][index_znota].values[0]
print(f"redshift = {redshift_legdr8} +/- {redshifterr_legdr8}")

In [ ]:
list_of_bands = df_f["r:band"].unique()
list_bands_ordered = []
for band in BAND_ORDER:
    if band in list_of_bands:
        list_bands_ordered.append(band)

In [ ]:
list(df_f.columns)

In [ ]:
df_r.columns

In [ ]:
df_r["date"]

In [ ]:
df_r["date"] = pd.to_datetime(df_r["date"])

In [ ]:
tr = Time(df_r["date"].values)
df_r["mjd"] = tr.mjd
df_r = df_r.sort_values("mjd")

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.errorbar(
    df_r.mjd,
    df_r.flux,
    yerr=df_r.flux_err,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="blue",
    label="recalc ap phot (Garazi)",
)

ax1.errorbar(
    df_r.mjd,
    df_r.psfFlux,
    yerr=df_r.psfFluxErr,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="red",
    label="Rubin psfFlux (DRP)",
)

ax1.errorbar(
    df_r.mjd,
    df_r.scienceFlux,
    yerr=df_r.scienceFluxErr,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="purple",
    label="Rubin scienceFlux(DRP)",
)


add_date_axis_on_top(ax1, df_r.mjd)
ax1.grid()
ax1.legend()

for band in list_bands_ordered:
    df = df_f[df_f["r:band"] == band]
    ax2.errorbar(
        df["r:midpointMjdTai"],
        df["r:psfFlux"],
        yerr=df["r:psfFluxErr"],
        fmt="o",  # <-- pas de "-"
        ms=4,
        lw=1.2,
        capsize=3,
        color=BAND_COLOR[band],
        label=f"Fink - {band}",
    )

ax2.grid()
ax2.set_xlabel("Date (MJD)", fontsize=12, labelpad=6)